# IMU Sensor Data Analysis and Motion Prediction
## Theoretical Foundations Notebook

**Author:** Prem M Patel  
**Affiliation:** Computer Engineering, Sankalchand Patel University  
**Contact:** prempatel7740@gmail.com

**Project:** Human Activity Recognition from Raw Inertial Measurement Unit Signals  
**Dataset:** UCI HAR — University of California Irvine Human Activity Recognition  
**Model:** CNN-LSTM Hybrid with Additive Attention Mechanism

---

> This notebook contains no executable code. It is a self-contained theoretical reference covering every concept, equation, and design decision used in the implementation notebook. Read this first. Every line of code in the implementation notebook directly realises one of the ideas developed here.

---

## Preface

The problem of inferring human activity from raw inertial sensor measurements sits at the intersection of signal processing, statistical estimation, and sequence modelling. It is deceptively simple to state — given a two-and-a-half-second window of accelerometer and gyroscope readings, determine whether the person was walking, sitting, standing, or lying down — and genuinely difficult to solve well, particularly when two of the six activity classes produce signals that are nearly indistinguishable by any classical measure.

This notebook develops the theoretical machinery required to understand every component of the solution: the physics of inertial measurement, the mathematics of frequency-domain analysis, the architecture and learning dynamics of convolutional and recurrent neural networks, the attention mechanism, and the statistical framework for evaluating multi-class classifiers.

The treatment is intentionally rigorous. Intuitions are provided alongside formal derivations, not as substitutes for them. A reader who works through this material carefully will not only understand why each architectural choice was made but will be equipped to improve upon it.

---

## 1. The Measurement Problem

### 1.1 What an Inertial Measurement Unit Measures

An Inertial Measurement Unit (IMU) is a device that measures the specific force and angular rate of a body in motion. In the context of this project, the IMU is a smartphone carried by a human subject, and the relevant outputs are:

**Accelerometer:** Measures specific force — the non-gravitational acceleration acting on the device. The output is a three-dimensional vector $\mathbf{a} = [a_x, a_y, a_z]^T$ with units of metres per second squared (m/s²). In practice, the accelerometer cannot distinguish between gravitational acceleration and kinematic acceleration, so the raw measurement is:

$$\tilde{\mathbf{a}} = \mathbf{a}_{kinematic} + \mathbf{g}$$

where $\mathbf{g}$ is the gravitational vector in the body frame. At rest on a flat surface, a perfectly calibrated accelerometer reads $\|\mathbf{g}\| = 9.81$ m/s² along the vertical axis and zero on the others. This gravitational component must be understood when interpreting the signal.

The UCI HAR dataset provides two versions of the accelerometer signal:
- **Total acceleration** $\tilde{\mathbf{a}}$ — raw measurement including gravity
- **Body acceleration** $\mathbf{a}_{body}$ — gravity-subtracted estimate, computed by the device's sensor fusion stack

**Gyroscope:** Measures angular velocity — the rate of rotation of the device about each of its three body axes. The output is $\boldsymbol{\omega} = [\omega_x, \omega_y, \omega_z]^T$ with units of radians per second (rad/s). The gyroscope provides no information about linear position or velocity, only instantaneous rotation rate.

### 1.2 The Sampling Model

The device samples both sensors at a rate of $f_s = 50$ Hz — fifty measurements per second. The Nyquist-Shannon sampling theorem establishes the fundamental constraint:

$$f_{Nyquist} = \frac{f_s}{2} = 25 \text{ Hz}$$

Any signal component with frequency above 25 Hz is aliased — it folds back into the representable band and cannot be distinguished from a genuine low-frequency component. The physical motions of human ambulation have characteristic frequencies well below 10 Hz, so $f_s = 50$ Hz is more than adequate for this application.

The dataset segments the continuous sensor stream into overlapping windows of $N = 128$ samples each, with 50% overlap between consecutive windows. Each window therefore spans:

$$T_{window} = \frac{N}{f_s} = \frac{128}{50} = 2.56 \text{ seconds}$$

This windowing scheme is the standard preprocessing step that converts a continuous time series into a supervised learning dataset with fixed-size inputs.

### 1.3 The Six Activity Classes

The dataset contains recordings of six activities performed by 30 volunteer subjects:

| Class Index | Activity | Physical Characteristics |
|:-----------:|:---------|:------------------------|
| 0 | WALKING | Periodic gait cycle, ~2 Hz dominant frequency |
| 1 | WALKING\_UPSTAIRS | Periodic but asymmetric gait, higher effort |
| 2 | WALKING\_DOWNSTAIRS | Periodic with impact loading, different phase |
| 3 | SITTING | Near-static, gravity dominant on one axis |
| 4 | STANDING | Near-static, gravity dominant, nearly identical to sitting |
| 5 | LAYING | Static, gravity on different axis than sitting/standing |

The fundamental challenge of this classification problem is the near-identical signal structure of SITTING and STANDING. Both activities produce a nearly static IMU signal with gravity acting predominantly along the same axis. The only discriminating information lies in subtle differences in body sway, postural micro-adjustments, and the precise orientation of the device — all of which produce very small signal variations buried within sensor noise.

---

## 2. Frequency-Domain Analysis of IMU Signals

### 2.1 The Discrete Fourier Transform

The time-domain representation of a signal — amplitude as a function of time — is the natural representation for plotting and visual inspection. However, many important properties of periodic and quasi-periodic signals are more naturally expressed in the frequency domain, where amplitude is expressed as a function of oscillation rate.

The Discrete Fourier Transform (DFT) of a finite sequence $x[0], x[1], \ldots, x[N-1]$ is defined as:

$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-j2\pi kn/N}, \quad k = 0, 1, \ldots, N-1$$

where $j = \sqrt{-1}$ is the imaginary unit, and each $X[k]$ is a complex number encoding both the amplitude and phase of the frequency component at:

$$f_k = \frac{k \cdot f_s}{N}$$

The magnitude spectrum $|X[k]|$ gives the strength of each frequency component, and the phase spectrum $\angle X[k]$ gives its phase offset. For real-valued signals, the DFT is conjugate-symmetric: $X[N-k] = X[k]^*$, so the second half of the spectrum is redundant. Only the first $N/2$ bins contain unique information, corresponding to frequencies from 0 to $f_{Nyquist}$.

The **frequency resolution** of the DFT — the smallest difference in frequency that can be distinguished — is:

$$\Delta f = \frac{f_s}{N} = \frac{50}{128} \approx 0.39 \text{ Hz}$$

This means the DFT can discriminate between oscillation rates separated by at least 0.39 Hz within a 2.56-second window.

The Fast Fourier Transform (FFT) is an algorithm that computes the DFT in $O(N \log N)$ operations rather than the $O(N^2)$ required by direct computation of the definition above. For $N = 128$, this represents a factor of approximately $128 / \log_2 128 \approx 18$ speedup.

### 2.2 Frequency Signatures of Human Activities

The accelerometer signal of a walking person contains a dominant oscillatory component at the stepping cadence. Normal human walking involves approximately 1.8 to 2.2 steps per second, corresponding to a cadence of 1.8 to 2.2 Hz. Each complete gait cycle (two steps) produces one full oscillation, so the body acceleration signal has a dominant frequency near 1.9 Hz in most subjects.

The FFT magnitude spectrum therefore shows a clear peak in the range of 1.5 to 3.0 Hz for all three walking activities, with the exact peak location varying by subject, terrain gradient, and walking speed.

Static activities (SITTING, STANDING, LAYING) produce no such periodic component. Their FFT spectra are essentially flat and low-amplitude, reflecting the absence of rhythmic motion. The DC component (bin $k=0$, corresponding to $f=0$) captures the signal mean — primarily the gravitational offset — and should be excluded when searching for the dominant oscillatory frequency.

This frequency-domain separability between dynamic and static activities is exactly the property that the first convolutional layers of the CNN exploit. A 1D convolution with a sinusoidal kernel at 2 Hz will produce a strong activation for walking windows and near-zero activation for sitting windows.

### 2.3 The Signal Processing Challenge

The cosine similarity between the mean body acceleration windows of SITTING and STANDING — computed in the exploration analysis — is approximately 0.97. This indicates that the two activities produce almost identical average signals. The distinction lies not in the mean behaviour of the signal, but in second-order statistics: the variance of body sway, the distribution of micro-movement amplitudes, and the precise orientation of the gravitational vector.

These properties are subtle enough that even the FFT fails to separate the two classes — their frequency spectra are indistinguishable. This is the reason deep learning is necessary here: the relevant discriminating features are not expressible as simple linear functions of the signal and require the hierarchical nonlinear feature extraction that convolutional and recurrent layers provide.

---

## 3. Convolutional Neural Networks for Time Series

### 3.1 The Convolutional Operation

A 1D convolutional layer applies a set of learnable filters to the input sequence. Given an input sequence $\mathbf{X} \in \mathbb{R}^{T \times C}$ with $T$ timesteps and $C$ channels, and a filter $\mathbf{W} \in \mathbb{R}^{k \times C \times F}$ with kernel size $k$ and $F$ output filters, the convolution operation computes:

$$\mathbf{Y}[t, f] = \sum_{i=0}^{k-1} \sum_{c=0}^{C-1} \mathbf{X}[t+i, c] \cdot \mathbf{W}[i, c, f] + b_f$$

for each output timestep $t$ and filter $f$. With `padding='same'`, the output has the same temporal length as the input. Without padding (`padding='valid'`), the output is shorter by $k-1$ timesteps.

The key insight is that the same filter weights are applied at every timestep — this is **weight sharing**, and it confers two important properties:

1. **Translation equivariance:** If the same pattern appears at different positions in the sequence, the same filter will detect it at each position. A footstep impact at timestep 30 activates the same filter as the same pattern at timestep 80.

2. **Parameter efficiency:** A filter of size $k$ applied to a sequence of length $T$ requires only $k$ parameters regardless of $T$. The same filter processes the entire sequence, unlike a fully-connected layer which requires $T$ separate weights.

### 3.2 Hierarchical Feature Extraction

The CNN architecture uses two blocks, each containing two convolutional layers followed by max pooling. This hierarchy is deliberate and theoretically motivated.

**Block 1 — Local feature detection (kernel receptive field: 60ms)**

Each filter in the first convolutional layer sees a window of $k=3$ consecutive timesteps. At 50 Hz, three timesteps span $3/50 = 60$ milliseconds. In this 60ms window, the filter learns to detect elementary signal features: a sharp positive spike (footstrike impact), a zero crossing (transition between phases), a brief plateau (momentary stillness). These are the atoms from which complex activity patterns are composed.

With 64 filters, Block 1 learns 64 distinct local pattern detectors. The output of Block 1 has shape $(T, 64)$ — at each timestep, 64 features describe the local signal neighbourhood.

**MaxPooling and the effective receptive field**

Max pooling with pool size 2 reduces the temporal dimension by half: $(128, 64) \rightarrow (64, 64)$. Crucially, each position in the pooled output corresponds to two positions in the input, so each filter in Block 2 effectively sees $2 \times k = 6$ original timesteps. The **effective receptive field** grows with depth:

$$r_{\ell} = r_{\ell-1} + (k-1) \cdot \prod_{i=1}^{\ell-1} s_i$$

where $r_\ell$ is the receptive field at layer $\ell$, $k$ is the kernel size, and $s_i$ is the stride of each preceding pooling layer. For our architecture:

- After Block 1, Layer 1: receptive field = 3 timesteps = 60ms
- After Block 1, Layer 2: receptive field = 5 timesteps = 100ms  
- After MaxPool 1: effective step covers 2 original timesteps
- After Block 2, Layer 1: effective receptive field = 9 original timesteps = 180ms
- After Block 2, Layer 2: effective receptive field = 13 original timesteps = 260ms
- After MaxPool 2: effective step covers 4 original timesteps

**Block 2 — Compositional feature detection (effective receptive field: 260ms)**

Block 2 uses 128 filters — double that of Block 1 — operating on the pooled output of Block 1. Each filter in Block 2 sees a 260ms window of the original signal, mediated through the features extracted by Block 1. At this scale, the model can detect multi-spike patterns that constitute partial gait cycles, the characteristic heel-to-toe loading pattern, or the sustained stillness of a static posture.

The output of Block 2 has shape $(32, 128)$ — 32 compressed timesteps, each described by 128 CNN-extracted features. This is the input to the LSTM.

### 3.3 The ReLU Activation Function

The Rectified Linear Unit activation function is defined as:

$$\text{ReLU}(x) = \max(0, x)$$

It is applied element-wise after each convolutional layer. Its role is to introduce nonlinearity into the model — without activation functions, a stack of linear transformations collapses to a single linear transformation regardless of depth, and the network loses all representational capacity beyond linear mappings.

ReLU is preferred over sigmoid and tanh in deep networks because it does not suffer from the vanishing gradient problem for positive activations: $d/dx \, \text{ReLU}(x) = 1$ for $x > 0$, so gradients propagate without attenuation through layers where the unit is active.

### 3.4 Batch Normalisation

Batch normalisation normalises the pre-activation values within each mini-batch:

$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$$

$$y_i = \gamma \hat{x}_i + \beta$$

where $\mu_B$ and $\sigma_B^2$ are the mini-batch mean and variance, $\epsilon$ is a small constant for numerical stability, and $\gamma$ and $\beta$ are learnable scale and shift parameters.

The normalisation counteracts **internal covariate shift** — the phenomenon where the distribution of layer inputs changes during training as the parameters of preceding layers are updated. By keeping the distribution of activations stable, batch normalisation allows higher learning rates and reduces sensitivity to weight initialisation. It also has a mild regularising effect that reduces the need for dropout.

### 3.5 Dropout as Regularisation

Dropout randomly sets a fraction $p$ of neuron outputs to zero during each training step. For a layer with output $\mathbf{h}$, the dropped output is:

$$\tilde{h}_i = \frac{m_i \cdot h_i}{1-p}, \quad m_i \sim \text{Bernoulli}(1-p)$$

The division by $(1-p)$ is the inverted dropout scaling that ensures the expected value of each neuron's output is unchanged between training and inference.

The regularising effect of dropout can be understood through the lens of ensemble learning. Each training step uses a different randomly sampled sub-network. Training with dropout is approximately equivalent to averaging the predictions of an exponentially large ensemble of sub-networks — a well-known bias-reducing technique. The network is forced to learn redundant representations that do not rely on the co-adaptation of specific pairs of neurons.

---

## 4. Long Short-Term Memory Networks

### 4.1 The Problem with Vanilla Recurrent Networks

A vanilla Recurrent Neural Network (RNN) maintains a hidden state $\mathbf{h}_t$ that summarises the information seen up to timestep $t$:

$$\mathbf{h}_t = \tanh(\mathbf{W}_h \mathbf{h}_{t-1} + \mathbf{W}_x \mathbf{x}_t + \mathbf{b})$$

During backpropagation through time (BPTT), gradients at timestep $t$ are computed by multiplying gradients at timestep $t+1$ by the weight matrix $\mathbf{W}_h$. Over a sequence of $T$ timesteps, the gradient at the earliest timestep involves the product:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{h}_0} = \prod_{t=1}^{T} \frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} \cdot \frac{\partial \mathcal{L}}{\partial \mathbf{h}_T}$$

Each factor $\partial \mathbf{h}_t / \partial \mathbf{h}_{t-1}$ involves the weight matrix $\mathbf{W}_h$ and the derivative of $\tanh$. If the spectral radius of $\mathbf{W}_h$ is less than 1, this product vanishes exponentially — the **vanishing gradient problem**. If it exceeds 1, it explodes. In either case, the network cannot learn dependencies spanning more than approximately 10 timesteps in practice.

For our application, the CNN output has 32 timesteps spanning 2.56 seconds. Dependencies between the beginning and end of the sequence are essential — the model must recognise that a series of footsteps observed over the full window constitutes walking, not just the most recent few steps.

### 4.2 The LSTM Architecture

Hochreiter and Schmidhuber (1997) introduced the Long Short-Term Memory architecture specifically to address the vanishing gradient problem. The key innovation is the **cell state** $\mathbf{c}_t$ — a separate memory pathway that can carry information across many timesteps with only additive interactions, enabling gradients to flow without attenuation.

The LSTM cell is governed by the following equations:

**Forget gate** — decides what information to discard from the cell state:
$$\mathbf{f}_t = \sigma(\mathbf{W}_f [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f)$$

**Input gate** — decides what new information to write to the cell state:
$$\mathbf{i}_t = \sigma(\mathbf{W}_i [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i)$$

**Cell candidate** — the new candidate values to be added to the cell state:
$$\tilde{\mathbf{c}}_t = \tanh(\mathbf{W}_c [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_c)$$

**Cell state update** — controlled combination of forgetting old state and writing new:
$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t$$

**Output gate** — decides what to output as the hidden state:
$$\mathbf{o}_t = \sigma(\mathbf{W}_o [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o)$$

**Hidden state output:**
$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$$

where $\sigma(\cdot)$ is the sigmoid function $\sigma(x) = 1/(1 + e^{-x})$, $\odot$ denotes element-wise multiplication, and $[\mathbf{h}_{t-1}, \mathbf{x}_t]$ denotes concatenation.

The notation $[\mathbf{h}_{t-1}, \mathbf{x}_t]$ makes explicit that all four gate weight matrices operate on the same concatenated vector, enabling each gate to condition its decision on both the previous hidden state and the current input.

### 4.3 Why the LSTM Resolves the Vanishing Gradient

The critical observation is in the cell state update equation:

$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t$$

The gradient of the loss with respect to the cell state at time $t-1$ is:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{c}_{t-1}} = \frac{\partial \mathcal{L}}{\partial \mathbf{c}_t} \cdot \mathbf{f}_t$$

This is a simple element-wise multiplication by the forget gate values — not a matrix multiplication by the weight matrix. When the forget gate is close to 1 (the network has learned to preserve information), gradients flow back through time with minimal attenuation. The cell state acts as a **gradient highway** that allows the network to learn dependencies across long sequences.

### 4.4 The LSTM in the Context of This Model

The LSTM receives the CNN output of shape $(32, 128)$ — a sequence of 32 compressed feature vectors, each encoding local motion patterns detected over approximately 80ms of the original signal. With `return_sequences=True`, the LSTM produces an output of shape $(32, 128)$ — a hidden state at each of the 32 timesteps.

These 32 hidden states are not equivalent. Early hidden states carry information about motion patterns at the beginning of the window; later states have processed more of the sequence and integrate longer-range dependencies. The attention mechanism, described in Section 5, exploits this non-equivalence to focus the final representation on the most informative timesteps.

The LSTM has four weight matrices, each of size $(d_h + d_x) \times d_h$ where $d_h = 128$ is the hidden dimension and $d_x = 128$ is the input dimension from the CNN. The total parameter count for the LSTM layer is approximately:

$$4 \times (128 + 128) \times 128 + 4 \times 128 = 131,072 + 512 = 131,584$$

This accounts for the largest fraction of the model's total parameter count.

---

## 5. The Attention Mechanism

### 5.1 Motivation

The standard approach in sequence classification is to take the final hidden state of the LSTM — $\mathbf{h}_{T}$ — as the sequence representation and pass it to the classifier. This is implicitly a strong assumption: that the last timestep summarises everything relevant about the sequence. For most sequences this is approximately true, because the LSTM accumulates information across timesteps and the final state reflects the entire history.

However, when the discriminating signal is concentrated in a small fraction of the sequence — as in the SITTING versus STANDING problem, where the key information may lie in only a few timesteps with elevated micro-movement — the final hidden state dilutes this information by integrating it with the many uninformative timesteps that surround it.

The attention mechanism addresses this by computing a **weighted sum** of all hidden states, where the weights are learned to assign greater influence to more informative timesteps.

### 5.2 Additive Attention (Bahdanau Attention)

The attention mechanism used in this model is additive attention, first introduced by Bahdanau, Cho, and Bengio (2015) in the context of neural machine translation.

Given the sequence of LSTM hidden states $\{\mathbf{h}_1, \mathbf{h}_2, \ldots, \mathbf{h}_T\}$ where $T = 32$ and $\mathbf{h}_t \in \mathbb{R}^{128}$, the attention mechanism proceeds in three steps:

**Step 1 — Score each hidden state:**
$$e_t = \mathbf{v}^T \tanh(\mathbf{W}_a \mathbf{h}_t + \mathbf{b}_a)$$

In the implementation, $\mathbf{W}_a \in \mathbb{R}^{128 \times 128}$ and $\mathbf{v} \in \mathbb{R}^{128}$ are the learnable parameters of the scoring function. Simplified to a single Dense(1) layer: $e_t = \mathbf{w}_a^T \tanh(\mathbf{h}_t) + b_a$ where $\mathbf{w}_a \in \mathbb{R}^{128}$.

The $\tanh$ activation constrains scores to $[-1, +1]$, providing bounded inputs to the subsequent softmax.

**Step 2 — Normalise scores to attention weights:**
$$\alpha_t = \frac{\exp(e_t)}{\sum_{s=1}^{T} \exp(e_s)}$$

The softmax transformation ensures the attention weights are non-negative and sum to one: $\sum_{t=1}^{T} \alpha_t = 1$. This is the formal constraint that makes them interpretable as a probability distribution over timesteps — a probability that each timestep is the "most relevant" position.

**Step 3 — Compute the context vector:**
$$\mathbf{c} = \sum_{t=1}^{T} \alpha_t \mathbf{h}_t$$

The context vector $\mathbf{c} \in \mathbb{R}^{128}$ is the attended representation of the sequence. Timesteps with high $\alpha_t$ contribute proportionally more to $\mathbf{c}$ than timesteps with low $\alpha_t$.

### 5.3 Why Attention Helps SITTING vs STANDING

Consider the physical difference between SITTING and STANDING. Both are static postures in which the subject is approximately at rest. The accelerometer measures near-zero body acceleration in both cases. However:

- When standing, the body's centre of mass undergoes small postural sway oscillations, typically in the range of 0.1 to 0.3 Hz with amplitude on the order of millimetres. These produce very small accelerometer deflections.
- When sitting, the support surface constrains the pelvis, reducing sway. The signal is marginally more static.

These differences are real but extremely subtle. In a 2.56-second window of 32 compressed LSTM states, the evidence for one posture versus the other may be concentrated in only 3 to 5 timesteps where a postural sway event occurred. A simple final-state classifier would dilute this evidence across all 32 states.

The attention mechanism learns to assign high weights $\alpha_t$ to precisely these informative timesteps and low weights to the uninformative majority. The resulting context vector $\mathbf{c}$ is a representation of the sequence that emphasises the few moments of discriminating postural activity.

This is the theoretical justification for the empirically observed result: STANDING recall improved from 76.9% (baseline without attention) to 80.6% (with attention), while SITTING performance was largely maintained.

### 5.4 Parameter Count of the Attention Layer

The attention mechanism — a single Dense(1) layer applied to the 128-dimensional LSTM output — requires:

$$128 \times 1 + 1 = 129 \text{ parameters}$$

This is a negligible fraction (0.06%) of the total model parameter count. The asymmetry is striking: 129 parameters produce a 3.7 percentage point improvement in the hardest activity class. This efficiency is characteristic of well-targeted inductive biases — small architectural changes that align the model's structure with the structure of the problem.

---

## 6. Training Methodology

### 6.1 Categorical Cross-Entropy Loss

For a $K$-class classification problem with one-hot encoded targets $\mathbf{y} \in \{0,1\}^K$ and predicted probability vector $\hat{\mathbf{y}} \in (0,1)^K$, the categorical cross-entropy loss is:

$$\mathcal{L}(\mathbf{y}, \hat{\mathbf{y}}) = -\sum_{k=1}^{K} y_k \log \hat{y}_k$$

Since exactly one component of $\mathbf{y}$ is non-zero (the true class $c$), this reduces to:

$$\mathcal{L} = -\log \hat{y}_c$$

This is the negative log-likelihood of the correct class. The loss is minimised when $\hat{y}_c \rightarrow 1$ — when the model assigns probability close to 1 to the true class. The logarithm creates a strong penalty for confident wrong predictions: if the true class is assigned probability 0.01, the loss is $-\log(0.01) \approx 4.6$, whereas if it is assigned probability 0.9, the loss is only $-\log(0.9) \approx 0.1$.

For the training set of $N = 7352$ windows, the total loss is the mean over samples:

$$\mathcal{L}_{total} = -\frac{1}{N} \sum_{n=1}^{N} \log \hat{y}_{c_n}$$

### 6.2 The Softmax Output Activation

The output layer produces $K = 6$ raw scores (logits) $z_1, z_2, \ldots, z_K$. The softmax activation converts these to a valid probability distribution:

$$\hat{y}_k = \frac{\exp(z_k)}{\sum_{j=1}^{K} \exp(z_j)}$$

The softmax function has two essential properties: all outputs are strictly positive, and they sum to exactly 1. Together, these ensure the output is a valid probability distribution over the six activity classes.

The combination of softmax output and cross-entropy loss has a convenient gradient structure. The gradient of the loss with respect to the logits is simply:

$$\frac{\partial \mathcal{L}}{\partial z_k} = \hat{y}_k - y_k$$

This is the difference between the predicted probability and the true label — the prediction error — making the gradient easy to compute and interpret.

### 6.3 The Adam Optimiser

Adam (Adaptive Moment Estimation, Kingma and Ba 2015) maintains per-parameter learning rates that are adapted based on estimates of the first and second moments of the gradients.

Let $g_t$ denote the gradient at timestep $t$. Adam maintains:

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t \quad \text{(first moment estimate)}$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2 \quad \text{(second moment estimate)}$$

The bias-corrected estimates are:

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

The parameter update is:

$$\theta_t = \theta_{t-1} - \frac{\alpha}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

Default values: $\alpha = 0.001$, $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$.

The adaptive learning rate has an important implication: parameters that receive large gradients (e.g., weights in the final dense layer that connect to the SITTING/STANDING confusion) automatically receive smaller effective learning rate updates, preventing oscillation. Parameters that receive small gradients receive larger effective updates, accelerating learning in under-trained directions.

### 6.4 Early Stopping and Learning Rate Reduction

Early stopping monitors the validation loss after each epoch and terminates training when the loss fails to improve by at least $\delta = 0.0005$ for $p = 12$ consecutive epochs, restoring the weights from the best epoch. This prevents overfitting beyond the optimal generalisation point.

ReduceLROnPlateau halves the learning rate when the validation loss fails to improve for $p = 5$ consecutive epochs. This allows the optimiser to make finer adjustments near the optimum, often recovering accuracy that was inaccessible at the higher learning rate.

### 6.5 The Train/Test Split and Generalisation

The UCI HAR dataset uses a **subject-based split**: 21 of the 30 subjects contribute to the training set, and the remaining 9 subjects contribute exclusively to the test set. No subject appears in both sets.

This is the correct evaluation protocol for an activity recognition system that will be deployed on new users. A random split — where windows from the same subject appear in both train and test sets — would be methodologically invalid, because the model could learn subject-specific idiosyncrasies (gait style, arm-swing pattern) rather than generalised activity signatures. The subject-based split forces the model to learn representations that generalise across individuals, which is the correct generalisation objective.

A model that achieves 93% accuracy under the subject-based split has demonstrated that it can recognise activities performed by people it has never observed during training. This is a substantially stronger claim than the same accuracy number under a random split.

---

## 7. Evaluation Metrics for Multi-Class Classification

### 7.1 The Confusion Matrix

The confusion matrix $\mathbf{C} \in \mathbb{Z}^{K \times K}$ has entry $C_{ij}$ equal to the number of samples whose true class is $i$ and whose predicted class is $j$. The diagonal entries $C_{ii}$ are correct predictions; off-diagonal entries are errors.

The normalised confusion matrix divides each row by its sum, yielding the fraction of samples from each true class that were assigned to each predicted class:

$$\tilde{C}_{ij} = \frac{C_{ij}}{\sum_{j'} C_{ij'}}$$

This is the per-class recall matrix. The diagonal $\tilde{C}_{ii}$ is the recall for class $i$.

### 7.2 Precision, Recall, and F1-Score

For each class $k$, define:

- **True Positives (TP):** $C_{kk}$ — correctly predicted as class $k$
- **False Positives (FP):** $\sum_{i \neq k} C_{ik}$ — incorrectly predicted as class $k$
- **False Negatives (FN):** $\sum_{j \neq k} C_{kj}$ — class $k$ samples predicted as something else

**Precision** measures the reliability of positive predictions:
$$P_k = \frac{TP_k}{TP_k + FP_k}$$

A model with high precision for STANDING rarely makes false STANDING predictions — when it says "STANDING", it is correct most of the time.

**Recall** measures the coverage of true positives:
$$R_k = \frac{TP_k}{TP_k + FN_k}$$

A model with high recall for STANDING correctly identifies most of the actual STANDING windows — it misses few.

**F1-score** is the harmonic mean of precision and recall:
$$F1_k = \frac{2 P_k R_k}{P_k + R_k} = \frac{2 TP_k}{2 TP_k + FP_k + FN_k}$$

The harmonic mean penalises imbalance between $P$ and $R$ more severely than the arithmetic mean. A model with $P = 1.0$ and $R = 0.5$ achieves $F1 = 0.667$, not $0.75$. This makes F1-score the appropriate single-number summary when both false positives and false negatives carry costs.

### 7.3 Macro and Weighted Averages

The **macro average** treats all classes equally regardless of support:
$$\bar{F1}_{macro} = \frac{1}{K} \sum_{k=1}^{K} F1_k$$

The **weighted average** weights each class by its support (number of samples):
$$\bar{F1}_{weighted} = \frac{\sum_{k=1}^{K} n_k \cdot F1_k}{\sum_{k=1}^{K} n_k}$$

For balanced datasets like UCI HAR, the two averages are nearly identical. The macro average is more sensitive to performance on minority classes — relevant when STANDING (532 test samples) is compared against LAYING (537 test samples).

### 7.4 The SITTING/STANDING Trade-off

The baseline model (without attention) exhibits a characteristic asymmetry:

- STANDING precision = 0.905, recall = 0.769
- SITTING precision = 0.781, recall = 0.902

The model was over-conservative about STANDING — it rarely predicted STANDING when uncertain, preferring to assign ambiguous windows to SITTING. This explains the high precision but low recall for STANDING, and the complementary pattern for SITTING.

With the attention mechanism, the model becomes more capable of identifying the subtle temporal features that distinguish STANDING. STANDING recall improved to 0.806, while SITTING precision improved to 0.806. The confusion between the two classes became more symmetric and overall less severe.

---

## 8. The UCI HAR Dataset

### 8.1 Collection Protocol

The UCI Human Activity Recognition dataset was compiled by Davide Anguita, Alessandro Ghio, Luca Oneto, Xavier Parra, and Jorge Luis Reyes-Ortiz at the Universitat Politecnica de Catalunya, with results published at the European Symposium on Artificial Neural Networks in 2013.

Thirty volunteer subjects aged 19 to 48 years performed each of the six activities while wearing a Samsung Galaxy S2 smartphone on the waist. The smartphone's built-in accelerometer and gyroscope were sampled at 50 Hz. Activity labels were obtained from manual video annotation.

The device orientation (waist-worn) is significant: with the phone carried at the waist, the dominant gravitational component in the total acceleration signal falls predominantly along the body's vertical axis (approximately the device's Y-axis), with smaller components on the other axes depending on the subject's posture and gait. This orientation is consistent across subjects because the device was standardised to a fixed waist position.

### 8.2 Pre-processing Applied by the Dataset Authors

The dataset authors applied the following preprocessing before publishing:

1. Noise filtering using a median filter and a 3rd order low-pass Butterworth filter at 20 Hz cutoff, applied to both accelerometer and gyroscope signals.

2. Gravitational separation: the gravitational component was estimated using a low-pass Butterworth filter at 0.3 Hz (since gravity is constant and thus a zero-frequency component relative to motion), and subtracted from the total acceleration to obtain the body acceleration.

3. Windowing: the filtered signals were segmented into 128-sample windows with 50% overlap (64 samples between consecutive window starts).

The implementation notebook applies StandardScaler normalisation on top of this pre-processed data, bringing each of the nine channels to zero mean and unit variance.

### 8.3 The Signal Channels

| Index | Channel | Description |
|:-----:|:--------|:-----------|
| 0 | body\_acc\_x | Body acceleration, X axis (gravity removed) |
| 1 | body\_acc\_y | Body acceleration, Y axis |
| 2 | body\_acc\_z | Body acceleration, Z axis |
| 3 | body\_gyro\_x | Angular velocity, X axis |
| 4 | body\_gyro\_y | Angular velocity, Y axis |
| 5 | body\_gyro\_z | Angular velocity, Z axis |
| 6 | total\_acc\_x | Total acceleration, X axis (gravity included) |
| 7 | total\_acc\_y | Total acceleration, Y axis |
| 8 | total\_acc\_z | Total acceleration, Z axis |

### 8.4 Benchmark Performance

The dataset is a standard benchmark in the human activity recognition literature. Published results on the subject-based test split range from approximately 89% for simple feature engineering with SVM to 94-96% for well-tuned deep learning models. The baseline CNN-LSTM model achieves 93.15% — competitive with published state-of-the-art on this split.

---

## 9. Relevance to Aerospace Inertial Systems

### 9.1 ISRO's Inertial Systems Unit

The Inertial Systems Unit (IISU) of the Indian Space Research Organisation, located in Thiruvananthapuram, develops and manufactures the inertial navigation systems used in all ISRO launch vehicles — PSLV, GSLV, LVM3, and the forthcoming Gaganyaan human spaceflight vehicle. The IISU's core products are ring laser gyroscopes and silicon MEMS accelerometers, and the flight computers that process their outputs in real time.

The signal processing and classification problem studied in this project, while applied to pedestrian activity recognition, is structurally identical to several problems that arise in aerospace inertial systems.

### 9.2 Structural Parallels

**Fault detection in flight IMU data**

A rocket's IMU produces accelerometer and gyroscope data continuously throughout the flight. Nominal flight follows a predictable trajectory envelope; sensor faults, structural failures, or aerodynamic anomalies produce signal deviations that must be detected and classified in real time. The CNN-LSTM architecture developed in this project — detecting local signal patterns with CNN layers and recognising temporal evolution with LSTM — is directly applicable to this fault detection problem. The six activity classes of the HAR problem are analogous to six flight states (nominal, sensor bias fault, gyroscope saturation, structural vibration, attitude error, system failure).

**Frequency analysis of structural vibrations**

Launch vehicles experience structural vibrations during ascent — from engine combustion, aerodynamic buffeting, and stage separation events. These vibrations have characteristic frequency signatures that must be monitored to ensure they remain within structural margins. The FFT analysis developed in Section 2 is the standard technique for characterising these frequency signatures from accelerometer data. The finding that walking activities have a 2 Hz dominant frequency and static activities have no dominant frequency is the same type of analysis applied to classify flight events.

**The attention mechanism in sequence modelling**

The attention mechanism's role in focusing on informative timesteps within a sequence has a direct analogue in anomaly detection for time series. An anomalous event in a flight trajectory — a brief attitude excursion, a transient sensor spike — may be confined to a few hundredths of a second within a longer data window. Standard LSTM classifiers that rely on the final hidden state average this brief event with the surrounding normal data, potentially missing it. An attention-augmented model learns to focus precisely on these brief anomalous intervals, improving detection sensitivity.

### 9.3 From Pedestrian to Spacecraft

The following table summarises the correspondence between the elements of this project and their aerospace analogues:

| This Project | ISRO Aerospace Application |
|:---|:---|
| Smartphone IMU at 50 Hz | IISU ring laser gyroscope + accelerometer |
| Six pedestrian activities | Six flight states or fault modes |
| CNN detects local motion patterns | CNN detects local signal anomalies |
| LSTM learns sequence of motion events | LSTM learns temporal flight trajectory |
| Attention focuses on postural sway events | Attention focuses on transient fault events |
| SITTING vs STANDING confusion | Nominal vs near-nominal flight confusion |
| Subject-based generalisation | Vehicle-to-vehicle generalisation |
| 93% classification accuracy | Detection rate with specified false alarm rate |

The mathematical content — convolutional feature extraction, recurrent sequence modelling, attention weighting, categorical cross-entropy training — is identical in both applications. The domain knowledge required to interpret the results and design the architecture differs, but the theoretical foundations developed in this notebook apply directly to the aerospace problem.

---

## 10. Summary of Theoretical Foundations

The following table provides a concise reference mapping each theoretical concept to its implementation location and its role in the overall system.

| Concept | Section | Implementation | Purpose |
|:--------|:-------:|:--------------|:--------|
| Nyquist sampling theorem | 1.2 | $f_s = 50$ Hz, window 128 samples | Establishes frequency resolution and aliasing limits |
| Windowing | 1.2 | 128-sample windows, 50% overlap | Converts continuous series to supervised learning dataset |
| Discrete Fourier Transform | 2.1 | `scipy.fft.fft` | Reveals frequency-domain activity signatures |
| Frequency resolution | 2.1 | $\Delta f = 50/128 = 0.39$ Hz | Determines discriminability of oscillation rates |
| Cosine similarity | 2.3 | Exploration analysis | Quantifies signal similarity between activity pairs |
| 1D convolution | 3.1 | `Conv1D(64, k=3)`, `Conv1D(128, k=3)` | Detects local temporal patterns in sensor signals |
| Effective receptive field | 3.2 | Hierarchical CNN blocks | Determines temporal scale of detected features |
| MaxPooling | 3.2 | `MaxPooling1D(2)` | Compresses sequence, expands receptive field |
| ReLU activation | 3.3 | `activation='relu'` | Introduces nonlinearity, prevents vanishing gradients |
| Batch normalisation | 3.4 | `BatchNormalization()` | Stabilises training, reduces internal covariate shift |
| Dropout | 3.5 | `Dropout(0.3)`, `Dropout(0.5)` | Ensemble regularisation, prevents memorisation |
| Vanishing gradient problem | 4.1 | Motivation for LSTM | Why vanilla RNN fails on 32-step sequences |
| LSTM gates | 4.2 | `LSTM(128, return_sequences=True)` | Long-range temporal dependency modelling |
| Cell state gradient highway | 4.3 | LSTM architecture | Enables gradient flow across 32 timesteps |
| Additive attention | 5.2 | `Dense(1, tanh)` + softmax + weighted sum | Focuses representation on informative timesteps |
| Context vector | 5.2 | `Lambda(K.sum, axis=1)` | Attended sequence summary for classifier |
| Categorical cross-entropy | 6.1 | `loss='categorical_crossentropy'` | Training objective for multi-class classification |
| Softmax | 6.2 | `activation='softmax'` | Converts logits to valid probability distribution |
| Adam optimiser | 6.3 | `Adam(lr=0.001)` | Adaptive per-parameter learning rate optimisation |
| Early stopping | 6.4 | `EarlyStopping(patience=12)` | Prevents overfitting beyond optimal generalisation |
| Subject-based split | 6.5 | 21/9 subject train/test | Correct generalisation protocol for new users |
| Confusion matrix | 7.1 | `confusion_matrix` | Complete error characterisation |
| Precision, recall, F1 | 7.2 | `precision_recall_fscore_support` | Per-class performance metrics |
| Macro/weighted average | 7.3 | `classification_report` | Aggregate performance summaries |

---

## References

Hochreiter, S. and Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*, 9(8), 1735–1780.

Bahdanau, D., Cho, K. and Bengio, Y. (2015). Neural machine translation by jointly learning to align and translate. *International Conference on Learning Representations (ICLR)*.

Kingma, D. P. and Ba, J. (2015). Adam: A method for stochastic optimisation. *International Conference on Learning Representations (ICLR)*.

Ioffe, S. and Szegedy, C. (2015). Batch normalisation: Accelerating deep network training by reducing internal covariate shift. *International Conference on Machine Learning (ICML)*.

Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I. and Salakhutdinov, R. (2014). Dropout: A simple way to prevent neural networks from overfitting. *Journal of Machine Learning Research*, 15, 1929–1958.

Anguita, D., Ghio, A., Oneto, L., Parra, X. and Reyes-Ortiz, J. L. (2013). A public domain dataset for human activity recognition using smartphones. *European Symposium on Artificial Neural Networks (ESANN)*.

LeCun, Y., Bengio, Y. and Hinton, G. (2015). Deep learning. *Nature*, 521, 436–444.

---

*This notebook constitutes the complete theoretical foundation for the IMU Sensor Data Analysis and Motion Prediction project. Every equation presented here is realised in executable code in the accompanying implementation notebook.*